# Linear drag: trajectory and range (Taylor 2.3)

*Class notebook, PHY 317.* Run each cell with Shift-Enter.

Eliminating $t$ from the linear-drag solutions ($y$ up) gives
$$y(x) = \frac{v_{y0} + v_{\rm ter}}{v_{x0}}\,x + v_{\rm ter}\tau\,\ln\!\left(1 - \frac{x}{v_{x0}\tau}\right),$$
with a vertical asymptote at $x = v_{x0}\tau$. The range $R$ solves $y(R) = 0$, which has no closed form; for small drag Taylor finds
$$R \approx R_{\rm vac}\left(1 - \frac{4}{3}\frac{v_{y0}}{v_{\rm ter}}\right), \qquad R_{\rm vac} = \frac{2 v_{x0} v_{y0}}{g}.$$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import brentq
from ipywidgets import interact

g = 9.8

def y_of_x(x, vx0, vy0, tau):
    vter = g * tau
    return (vy0 + vter) / vx0 * x + vter * tau * np.log(1 - x / (vx0 * tau))

def range_exact(vx0, vy0, tau):
    xmax = vx0 * tau * (1 - 1e-9)
    return brentq(lambda x: y_of_x(x, vx0, vy0, tau), 1e-9 * xmax, xmax)

## Move the sliders

Small $\tau$ means big drag. The dotted vertical line is the asymptote $x = v_{x0}\tau$; the dashed curve is the same launch with no drag.

In [ ]:
@interact(vx0=(1, 40, 1), vy0=(1, 40, 1), tau=(0.5, 30.0, 0.5))
def trajectory(vx0=20, vy0=15, tau=5.0):
    R_vac = 2 * vx0 * vy0 / g
    R = range_exact(vx0, vy0, tau)
    R_approx = R_vac * (1 - 4 * vy0 / (3 * g * tau))

    xv = np.linspace(0, R_vac, 300)
    x = np.linspace(0, R, 300)
    plt.figure(figsize=(8, 4))
    plt.plot(xv, vy0 / vx0 * xv - g * xv**2 / (2 * vx0**2), "k--", label="no drag")
    plt.plot(x, y_of_x(x, vx0, vy0, tau), label="linear drag")
    plt.axvline(vx0 * tau, ls=":", color="gray")
    plt.xlim(0, R_vac * 1.05); plt.ylim(0, None)
    plt.xlabel("x (m)"); plt.ylabel("y (m)"); plt.legend(); plt.grid(alpha=0.3)
    plt.show()
    print(f"R_vac = {R_vac:6.1f} m    R exact = {R:6.1f} m    Taylor's approximation = {R_approx:6.1f} m")

## How good is the small-drag approximation?

Exact range against Taylor's one-term formula, as a function of $v_{y0}/v_{\rm ter}$.

In [ ]:
vx0, vy0 = 20, 15
taus = np.logspace(0, 2, 40)
R_vac = 2 * vx0 * vy0 / g
exact = [range_exact(vx0, vy0, t) / R_vac for t in taus]
approx = [1 - 4 * vy0 / (3 * g * t) for t in taus]

plt.semilogx(vy0 / (g * taus), exact, label="exact")
plt.semilogx(vy0 / (g * taus), approx, "--", label="1 - (4/3) v_y0/v_ter")
plt.xlabel("v_y0 / v_ter"); plt.ylabel("R / R_vac"); plt.ylim(0, 1.05)
plt.legend(); plt.grid(alpha=0.3, which="both"); plt.show()